# 04 — Carbon Mapping & District Summary
**Forest Carbon Stock Estimation — Nainital District, Uttarakhand**

This notebook covers:
- Wall-to-wall biomass prediction across all forest pixels
- IPCC carbon and CO₂e conversion
- Sequestration potential classification
- Four-panel output map
- District-level summary statistics
- Biomass vs elevation transect analysis
- Interactive Folium map

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import yaml
import joblib
import folium
from pathlib import Path

from src.models.predict import classify_seq_potential, write_raster_from_points

with open('../config/config.yaml') as f:
    CFG = yaml.safe_load(f)
PATHS = CFG['paths']
FEAT  = CFG['features']
IPCC  = CFG['ipcc']
SEQ   = CFG['seq_potential']
VIZ   = CFG['viz']

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120})
print('Setup complete.')

## 1. Load Model & Prediction Grid

In [ ]:
model = joblib.load(Path('..') / 'outputs/models/xgb_model.joblib')
print('Model loaded:', type(model).__name__)

df = pd.read_csv(Path('..') / PATHS['prediction_grid'])
print(f'Prediction grid: {len(df):,} forest pixels')
df.head(3)

## 2. Predict & Convert

In [ ]:
X = df[FEAT['model_features']].values
biomass = np.clip(model.predict(X), 0, None)

df['biomass_tpha']  = biomass
df['carbon_tcha']   = biomass * IPCC['biomass_to_carbon']
df['co2e_tco2eha']  = df['carbon_tcha'] * IPCC['carbon_to_co2e']
df['seq_zone']      = classify_seq_potential(
    biomass   = df['biomass_tpha'].values,
    ndvi      = df['NDVI'].values,
    slope     = df['slope'].values,
    elevation = df['elevation'].values
)

print(f"Biomass  — mean: {df['biomass_tpha'].mean():.1f}  max: {df['biomass_tpha'].max():.1f} t/ha")
print(f"Carbon   — mean: {df['carbon_tcha'].mean():.1f} tC/ha")
print(f"CO₂e     — mean: {df['co2e_tco2eha'].mean():.1f} tCO₂e/ha")

## 3. Biomass Distribution by Elevation Band

In [ ]:
bins   = [0, 500, 1000, 1500, 2000, 2500, 3000, 9999]
labels = ['<500','500-1000','1000-1500','1500-2000','2000-2500','2500-3000','>3000']
df['elev_band'] = pd.cut(df['elevation'], bins=bins, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean biomass by elevation band
summary = df.groupby('elev_band', observed=True)['biomass_tpha'].agg(['mean','std','count'])
axes[0].bar(summary.index, summary['mean'], yerr=summary['std'],
             color='#4caf50', edgecolor='white', capsize=4)
axes[0].set_xlabel('Elevation Band (m)'); axes[0].set_ylabel('Mean AGBD (t/ha)')
axes[0].set_title('Mean Biomass by Elevation Band', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].spines[['top','right']].set_visible(False)

# Scatter: biomass vs elevation (sampled)
samp = df.sample(min(5000, len(df)), random_state=42)
sc = axes[1].scatter(samp['elevation'], samp['biomass_tpha'],
                      c=samp['NDVI'], cmap='YlGn', alpha=0.3, s=5)
plt.colorbar(sc, ax=axes[1], label='NDVI')
axes[1].set_xlabel('Elevation (m)'); axes[1].set_ylabel('Predicted AGBD (t/ha)')
axes[1].set_title('Biomass vs Elevation (coloured by NDVI)', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Four-Panel Output Map

In [ ]:
from src.visualization.maps import map_four_panel
fig = map_four_panel(save=True)
plt.show()

## 5. District Summary Statistics

In [ ]:
PIXEL_HA = (20 * 20) / 10_000   # 20 m pixel → 0.04 ha

forest_ha   = len(df[df['biomass_tpha'] > 0]) * PIXEL_HA
total_c_mtc = (df['carbon_tcha'].sum() * PIXEL_HA) / 1e6
total_co2_m = (df['co2e_tco2eha'].sum() * PIXEL_HA) / 1e6

summary = {
    'District':                    'Nainital',
    'State':                       'Uttarakhand',
    'Forest Area (km²)':           round(forest_ha / 100, 1),
    'Forest Area (ha)':            round(forest_ha, 0),
    'Mean Biomass (t/ha)':         round(df['biomass_tpha'].mean(), 1),
    'Median Biomass (t/ha)':       round(df['biomass_tpha'].median(), 1),
    'Mean Carbon Stock (tC/ha)':   round(df['carbon_tcha'].mean(), 1),
    'Total Carbon Stock (MtC)':    round(total_c_mtc, 2),
    'Total CO₂e Stored (MtCO₂e)': round(total_co2_m, 2),
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
summary_df.style.set_properties(**{'background-color': '#f1f8e9', 'color': '#1a1a1a'})

## 6. Sequestration Potential Summary

In [ ]:
zone_labels = SEQ['zones']
zone_colors = VIZ['seq_colors']

zone_df = df[df['seq_zone'] > 0].groupby('seq_zone').agg(
    Pixels=('seq_zone', 'count'),
    Area_ha=('seq_zone', lambda x: len(x) * PIXEL_HA),
    Mean_NDVI=('NDVI', 'mean'),
    Mean_Slope=('slope', 'mean'),
    Mean_Elevation=('elevation', 'mean')
).reset_index()
zone_df['Zone Label'] = zone_df['seq_zone'].map(zone_labels)
print(zone_df.to_string(index=False))

# Pie chart
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    zone_df['Area_ha'],
    labels=[f"Zone {z}: {zone_labels[z]}" for z in zone_df['seq_zone']],
    colors=zone_colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
ax.set_title('Carbon Sequestration Potential\nArea Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Interactive Map (Folium)

In [ ]:
from src.visualization.maps import make_interactive_map
m = make_interactive_map(save=True)
m    # renders inline in notebook

## 8. Export District Summary CSV

In [ ]:
stat_dir = Path('..') / 'outputs' / 'stats'
stat_dir.mkdir(parents=True, exist_ok=True)
out = stat_dir / 'district_summary.csv'
pd.DataFrame([summary]).to_csv(out, index=False)
print(f'Saved: {out}')